# Assignment 4: Proposal


## Link 
https://youtu.be/ZZFsY06V7Ck


## Dataset

I'm using the same Twitch Gamers social network from SNAP:  https://snap.stanford.edu/data/twitch_gamers.html. 
Its a mutual follower network of Twitch streamers collected from the public API in 2018 with 168k nodes and 6M edges. Each node is a streamer and each edge means two streamers follow each other back. Alongside with the edge list the dataset also includes a features file with one row per streamer that has information like the view count, account age, language, and two categorical flags which we could use 

1.whether the streamer is Twitch affiliated and 

2.whether their content is flagged mature.

## Categorical variable

Ill use affiliate (yes/no) as the main categorical variable. Twitch affiliate status is only granted once a streamer hits certain audience and engagement thresholds, so it works like a realworld outcome tied to the network and not just an arbitrary label.

## Loading 

The full graph is too large to analyze directly so I'll take a smaller connected sample using a breadth first snowball search this time starting from one streamer walk outward through real follower edges until i get to 500. This keeps the sample as one connected neighborhood rather than scattered disconnected fragments. Ill then merge in the features file by streamer ID to attach the affiliate label to each sampled node.

## Possible questions

- Who are the most important streamers in this network and does that depend on which centrality measure you use? A streamer with the most followers wchich would be the degree isnt necessarily the same as a streamer connected to other well connected streamers ie eigenvector.
- Does affiliate status line up with network position or is it driven by something else entirely like views or account age 
- Are there streamers who look unimportant by degree but turn out to be well positioned by eigenvector centrality or vice versa

## Analysis

- Compute degree centrality and eigenvector centrality for every node in the sample.
- Calculate basic summary stats for each measure, both overall and split by affiliate status.
- Compare the two groups using a t-test to check whether any difference in average centrality is statistically significant or just noise.
- Look at the top ranked nodes under each centrality measure side by side, to see whether they're the same streamers or different ones.

## Hypothesis

Im assuming degree centrality would differ  between affiliate and non affiliate streamers since affiliate status depends on reaching an audience and audience reach possibly correlates with how many other streamers follow you back.


In [2]:
from collections import defaultdict, deque

import pandas as pd
import networkx as nx
from scipy import stats

edges = pd.read_csv("large_twitch_edges.csv")
features = pd.read_csv("large_twitch_features.csv").set_index("numeric_id")

adjacency = defaultdict(list)
for a, b in zip(edges["numeric_id_1"], edges["numeric_id_2"]):
    adjacency[a].append(b)
    adjacency[b].append(a)

start_node = 12345
sample_ids = {start_node}
queue = deque([start_node])
while queue and len(sample_ids) < 500:
    node = queue.popleft()
    for neighbor in adjacency[node]:
        if neighbor not in sample_ids:
            sample_ids.add(neighbor)
            queue.append(neighbor)
            if len(sample_ids) >= 500:
                break

sample_edges = edges[edges["numeric_id_1"].isin(sample_ids) & edges["numeric_id_2"].isin(sample_ids)]
G = nx.from_pandas_edgelist(sample_edges, "numeric_id_1", "numeric_id_2")

sample_features = features.loc[list(sample_ids)]
print(sample_features["affiliate"].value_counts())

Affiliate counts:
affiliate
0    269
1    231
Name: count, dtype: int64


In [5]:
degree = dict(G.degree())

affiliate_degrees = [degree[n] for n in sample_ids if sample_features.loc[n, "affiliate"] == 1]
non_affiliate_degrees = [degree[n] for n in sample_ids if sample_features.loc[n, "affiliate"] == 0]

t_stat, p_value = stats.ttest_ind(affiliate_degrees, non_affiliate_degrees, equal_var=False)

print(f"t: {t_stat:.2f}")
print(f"p: {p_value:.2e}")

t: -3.55
p: 4.44e-04


## Expectations 

What I expect this to tell us is whether affiliate streamers are just well followed or whether theyre actually embedded in like a well connected core of other high status streamers. Degree centrality should answer how many direct connections a node has and eigenvector centrality should answer how connected those connections are. So comparing the two should show something a little more interesting. If affiliates score high on both measures that suggests network position really does track with Twitchs own criteria on affiliates. If they only score high on one that says something more specific about what kind of connectivity actually matters for getting noticed on the platform.